In [48]:
#Input monodromy decomposition. Should be an array of even length with only positive integers.
twist = [1,1,1,1]


#Number of tetrahedra in layer-triangulation
period = 0

for i in twist:
    period = period + abs(i)


#Compute psl-obstruction classes
#According to Lefshetz duality, the relative cohomology H2 (M, boundary(M)) is isomorphic to the homology H1(M). 
#The fundamental group of M is the semidirect product Z X F2. Where tx=phi(x)t, ty=phi(y)t. Abelianizing gives <t,x,y|x=phi(x),y=phi(y)>.
#We will therefore first compute phi(x) and phi(y).
R = Matrix([[1,1],[0,1]])
L = Matrix([[1,0],[1,1]])

#We construct all intermediate matrices in the triangulation which will be useful later.
M=[matrix.identity(2)]
for i in range(len(twist)):
    for j in range (abs(twist[i])):
        if (i % 2) == 0:
            M.append(M[-1] * R^math.copysign(1, twist[i]))
        else:
            M.append(M[-1] * L^math.copysign(1, twist[i]))


#Here we determine the relators for the homology. If they are cyclic with even order, then they will not generate an obstruction class.
N = (M[-1]-matrix.identity(2)).echelon_form()

if N[0][1] == 0:
    relators = [N[0][0], N[1][1]]
else:
    relators = [N[0][0] * lcm(N[1][1],N[0][1]) / N[0][1], N[1][1]]

redrelators = [relators[0] % 2, relators[1] % 2]


#We construct the trivial obstruction class by restricting each cocycle to each simplex and finding the respective edges for which it is a coboundary.
obs_triv = []
for i in range(period):
    obs_triv.append([1,1,1,1,1,1])


#We construct the obstruction class corresponding to the meridian of the periferal subgroup.
obs_fund = []
for i in range(len(twist)):
    for j in range(twist[i]):
        if (i % 2) == 0:
            obs_fund.append([1,-1,1,1,1,1])
        else:
            obs_fund.append([1,1,1,1,1,-1])

            
#We construct the obstruction class corresponding to each surface group generator:
obs_mer = [[1,1,-1,1,1,1],[1,1,1,-1,1,1]]
for i in range(period-2):
    obs_mer.append([1,1,1,1,1,1])

obs_long = [[1,1,1,-1,1,1],[1,1,-1,1,1,1]]
for i in range(period-2):
    obs_long.insert(i+1, [1,1,1,1,1,1])

obstruction = [obs_triv, obs_fund]
if (redrelators[0] != 0) & (relators[0] != 1):
    obstruction.append(obs_mer)
if (redrelators[1] != 0) & (relators[1] != 1):
    obstruction.append(obs_long)
    


#We are now ready to construct the Ptolemy equations


#First prepare unreduced edge-equations for the Ptolemy Variety
gens = 'y01_0, y02_0, y03_0, y12_0, y13_0, y23_0'
for i in range(period - 1):
    gens = gens + ' ,'+'y01_{index}, y02_{index}, y03_{index}, y12_{index}, y13_{index}, y23_{index}'.format( index = i + 1 )
    
R=PolynomialRing(QQbar, 6*period, gens)
R.inject_variables()



#We prepare unreduced Ptolemy equations
ideals = []
# varieties = []

for obs in obstruction:
    unred_ptolemy = []
    for i in range(period):
        unred_ptolemy.append(f'y03_{i} * y12_{i} + y01_{i} * y23_{i} - y02_{i} * y13_{i}')
        
    unred_ptolemy.append(f'{obs[period-1][2]*obs[0][0]} * y03_{period-1}-y01_{0}')
    unred_ptolemy.append(f'{obs[period-1][2]*obs[0][5]} * y03_{period-1}-y23_{0}')
    unred_ptolemy.append(f'{obs[period-1][4]*obs[0][4]} * y13_{period-1}-y13_{0}')
    unred_ptolemy.append(f'{obs[period-1][1]*obs[0][1]} * y02_{period-1}-y02_{0}')   
    unred_ptolemy.append(f'{obs[period-1][0]*obs[0][3]} * y01_{period-1}+y12_{0}')
    unred_ptolemy.append(f'{obs[period-1][5]*obs[0][3]} * y23_{period-1}+y12_{0}')
    
    k=0
    for i in range(len(twist)):
        for j in range(twist[i]):
            if (i % 2) == 1:
                unred_ptolemy.append(f'{obs[k][2]*obs[k+1][0]} * y03_{k}-y01_{k+1}')
                unred_ptolemy.append(f'{obs[k][2]*obs[k+1][5]} * y03_{k}-y23_{k+1}')
                unred_ptolemy.append(f'{obs[k][4]*obs[k+1][4]} * y13_{k}-y13_{k+1}')
                unred_ptolemy.append(f'{obs[k][1]*obs[k+1][1]} * y02_{k}-y02_{k+1}')   
                unred_ptolemy.append(f'{obs[k][0]*obs[k+1][3]} * y01_{k}+y12_{k+1}')
                unred_ptolemy.append(f'{obs[k][5]*obs[k+1][3]} * y23_{k}+y12_{k+1}')  
            else:
                unred_ptolemy.append(f'{obs[k][2]*obs[k+1][1]} * y03_{k}-y02_{k+1}')
                unred_ptolemy.append(f'{obs[k][2]*obs[k+1][4]} * y03_{k}-y13_{k+1}')
                unred_ptolemy.append(f'{obs[k][0]*obs[k+1][0]} * y01_{k}-y01_{k+1}')
                unred_ptolemy.append(f'{obs[k][5]*obs[k+1][5]} * y23_{k}-y23_{k+1}')
                unred_ptolemy.append(f'{obs[k][1]*obs[k+1][3]} * y02_{k}-y12_{k+1}')
                unred_ptolemy.append(f'{obs[k][4]*obs[k+1][3]} * y13_{k}-y12_{k+1}')
             
            k += 1 
            
            if k >= period - 1:
                break
            
            
        if k >= period - 1:
            break
            
    unred_ptolemy.append('y12_0-1')
    
    ideals.append(R.ideal(unred_ptolemy))
#     varieties.append(R.ideal(unred_ptolemy).variety())
  
# Establish the gluing set so that we can map the unreduced ideal to the reduced ideal.
gluings_in = []
gluings_out = []

for obs in obstruction:
    glue_in = []
    glue_out = []
    
    k=0
    for i in range(len(twist)):
        for j in range(twist[i]):
            if (i % 2) == 1:
                
                glue_in.append(f'y03_{k}')
                glue_out.append(f'{obs[k][2]*obs[k+1][0]} * y01_{k+1}')

                glue_in.append(f'y03_{k}')
                glue_out.append(f'{obs[k][2]*obs[k+1][5]} * y23_{k+1}')

                glue_in.append(f'y13_{k}')
                glue_out.append(f'{obs[k][4]*obs[k+1][4]} * y13_{k+1}')

                glue_in.append(f'y02_{k}')   
                glue_out.append(f'{obs[k][1]*obs[k+1][1]} * y02_{k+1}')

                glue_in.append(f'y01_{k}')
                glue_out.append(f'-{obs[k][0]*obs[k+1][3]} * y12_{k+1}')

                glue_in.append(f'y23_{k}')
                glue_out.append(f'-{obs[k][5]*obs[k+1][3]} * y12_{k+1}')
                
                 
            else:
                
                glue_in.append(f'y03_{k}')
                glue_out.append(f'{obs[k][2]*obs[k+1][1]} * y02_{k+1}')

                glue_in.append(f'y03_{k}')
                glue_out.append(f'{obs[k][2]*obs[k+1][4]} * y13_{k+1}')

                glue_in.append(f'y01_{k}')
                glue_out.append(f'{obs[k][0]*obs[k+1][0]} * y01_{k+1}')

                glue_in.append(f'y23_{k}')   
                glue_out.append(f'{obs[k][5]*obs[k+1][5]} * y23_{k+1}')

                glue_in.append(f'y02_{k}')
                glue_out.append(f'{obs[k][1]*obs[k+1][3]} * y12_{k+1}')

                glue_in.append(f'y13_{k}')
                glue_out.append(f'{obs[k][4]*obs[k+1][3]} * y12_{k+1}')
             
            k += 1 
            
            if k >= period - 1:
                break
            
            
        if k >= period - 1:
            break
    glue_in.append(f'y03_{period-1}')
    glue_out.append(f'{obs[period-1][2]*obs[0][0]} * y01_{0}')
    
    glue_in.append(f'y03_{period-1}')
    glue_out.append(f'{obs[period-1][2]*obs[0][5]} * y23_{0}')
    
    glue_in.append(f'y13_{period-1}')
    glue_out.append(f'{obs[period-1][4]*obs[0][4]} * y13_{0}')
    
    glue_in.append(f'y02_{period-1}')   
    glue_out.append(f'{obs[period-1][1]*obs[0][1]} * y02_{0}')
    
    glue_in.append(f'y01_{period-1}')
    glue_out.append(f'-{obs[period-1][0]*obs[0][3]} * y12_{0}')
    
    glue_in.append(f'y23_{period-1}')
    glue_out.append(f'-{obs[period-1][5]*obs[0][3]} * y12_{0}')
    
    gluings_in.append(glue_in)
    gluings_out.append(glue_out)


#Euler Characteristic = 0 implies number of edges = number of tetrahedra
#Prepare edge-variables for reduced Ptolemy Variety

gens = 'x0'
for i in range(period - 1):
    gens = gens + ' ,'+'x{}'.format( i + 1 )
S=PolynomialRing(QQbar, period, gens)
S.inject_variables()

#Establish the homomorphisms between the rings
    
for obs in obstruction:
    hom = []
    for i in range(6*period):
        hom.append(0)
        
    
    
    
#     k=0
#     for i in range(len(twist)):
#         for j in range(twist[i]):
#             if (i % 2) == 1:
                
#                 glue_in.append(f'y03_{k}')
#                 glue_out.append(f'{obs[k][2]*obs[k+1][0]} * y01_{k+1}')

#                 glue_in.append(f'y03_{k}')
#                 glue_out.append(f'{obs[k][2]*obs[k+1][5]} * y23_{k+1}')

#                 glue_in.append(f'y13_{k}')
#                 glue_out.append(f'{obs[k][4]*obs[k+1][4]} * y13_{k+1}')

#                 glue_in.append(f'y02_{k}')   
#                 glue_out.append(f'{obs[k][1]*obs[k+1][1]} * y02_{k+1}')

#                 glue_in.append(f'y01_{k}')
#                 glue_out.append(f'-{obs[k][0]*obs[k+1][3]} * y12_{k+1}')

#                 glue_in.append(f'y23_{k}')
#                 glue_out.append(f'-{obs[k][5]*obs[k+1][3]} * y12_{k+1}')
                
                 
#             else:
                
#                 glue_in.append(f'y03_{k}')
#                 glue_out.append(f'{obs[k][2]*obs[k+1][1]} * y02_{k+1}')

#                 glue_in.append(f'y03_{k}')
#                 glue_out.append(f'{obs[k][2]*obs[k+1][4]} * y13_{k+1}')

#                 glue_in.append(f'y01_{k}')
#                 glue_out.append(f'{obs[k][0]*obs[k+1][0]} * y01_{k+1}')

#                 glue_in.append(f'y23_{k}')   
#                 glue_out.append(f'{obs[k][5]*obs[k+1][5]} * y23_{k+1}')

#                 glue_in.append(f'y02_{k}')
#                 glue_out.append(f'{obs[k][1]*obs[k+1][3]} * y12_{k+1}')

#                 glue_in.append(f'y13_{k}')
#                 glue_out.append(f'{obs[k][4]*obs[k+1][3]} * y12_{k+1}')
             
#             k += 1 
            
#             if k >= period - 1:
#                 break
            
            
#         if k >= period - 1:
#             break
#     glue_in.append(f'y03_{period-1}')
#     glue_out.append(f'{obs[period-1][2]*obs[0][0]} * y01_{0}')
    
#     glue_in.append(f'y03_{period-1}')
#     glue_out.append(f'{obs[period-1][2]*obs[0][5]} * y23_{0}')
    
#     glue_in.append(f'y13_{period-1}')
#     glue_out.append(f'{obs[period-1][4]*obs[0][4]} * y13_{0}')
    
#     glue_in.append(f'y02_{period-1}')   
#     glue_out.append(f'{obs[period-1][1]*obs[0][1]} * y02_{0}')
    
#     glue_in.append(f'y01_{period-1}')
#     glue_out.append(f'-{obs[period-1][0]*obs[0][3]} * y12_{0}')
    
#     glue_in.append(f'y23_{period-1}')
#     glue_out.append(f'-{obs[period-1][5]*obs[0][3]} * y12_{0}')




Defining y01_0, y02_0, y03_0, y12_0, y13_0, y23_0, y01_1, y02_1, y03_1, y12_1, y13_1, y23_1, y01_2, y02_2, y03_2, y12_2, y13_2, y23_2, y01_3, y02_3, y03_3, y12_3, y13_3, y23_3
Defining z0, z1, z2, z3, z4, z5, z6, z7, z8, z9, z10, z11, z12, z13, z14, z15, z16, z17, z18, z19, z20, z21, z22, z23
Defining x0, x1, x2, x3


y01_0

In [63]:
len(varieties[1])

6

In [65]:
varieties[1][5][y03_0].minpoly()

x^2 - 1/2*x + 1/2